# Dynamic-k Comparison Notebook

Run this notebook top to bottom in the online IDE. It does not require opening a terminal. The cells call the project runner from Python, compare the baseline methods against the two stochastic dynamic-k methods, and display the saved results and plots.

The default path is conservative: first run a tiny smoke comparison. After that succeeds, set `RUN_REAL = True` in the real-run cell to use the default Qwen models on the 40 GB GPU.

In [ ]:
from __future__ import annotations

import csv
import os
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, display

# Set this only if your notebook kernel starts outside the repository.
REPO_ROOT_OVERRIDE = ""


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "main.py").exists() and (candidate / "research").exists():
            return candidate
    raise RuntimeError("Could not find the repository root. Set REPO_ROOT_OVERRIDE.")


REPO_ROOT = Path(REPO_ROOT_OVERRIDE).resolve() if REPO_ROOT_OVERRIDE else find_repo_root(Path.cwd())
SCRIPT = REPO_ROOT / "research" / "v.poponnikov" / "notebooks" / "dynamic_k_comparison.py"
RESULT_DIR = REPO_ROOT / "research" / "v.poponnikov" / "results" / "dynamic_k_comparison"
PLOTS_DIR = REPO_ROOT / "research" / "v.poponnikov" / "plots" / "dynamic_k_comparison"
EXPERIMENTS = ["01_baseline", "08_+speedup_adapt", "stochastic_consensus_k", "latent_regime_k"]

print(f"Repository: {REPO_ROOT}")
print(f"Comparison script: {SCRIPT}")

## Helper for Cell-Based Runs

This helper runs Python commands with `PYTHONPATH=src` and streams output into the notebook cell.

In [ ]:
def run_python(*args: object) -> None:
    env = os.environ.copy()
    env["PYTHONPATH"] = str(REPO_ROOT / "src")
    env.setdefault("HF_HOME", str(REPO_ROOT / ".hf-cache"))
    env.setdefault("TOKENIZERS_PARALLELISM", "false")

    command = [sys.executable, *[str(arg) for arg in args]]
    print("Running:", " ".join(command))
    process = subprocess.Popen(
        command,
        cwd=REPO_ROOT,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Command failed with exit code {return_code}")

## Optional Dependency Install

If the online IDE image already has the project dependencies, leave this disabled. If imports fail later, set `INSTALL_DEPENDENCIES = True` and run the cell.

In [ ]:
INSTALL_DEPENDENCIES = False

if INSTALL_DEPENDENCIES:
    run_python("-m", "pip", "install", "-e", ".[dev,viz]")
else:
    print("Dependency install skipped.")

## GPU Check

In [ ]:
try:
    import torch

    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print("GPU:", props.name)
        print("VRAM GB:", round(props.total_memory / 1024**3, 2))
except ModuleNotFoundError:
    print("Torch is not installed in this kernel. Enable the dependency install cell above.")

## Confirm Research Experiments Are Discoverable

In [ ]:
run_python(REPO_ROOT / "src" / "main.py", "--list", "--research")

## Tiny Smoke Comparison

This runs all four comparison experiments with small OPT models. Use this first to check that the online IDE can download models, load datasets, and write plots.

In [ ]:
RUN_SMOKE = True
SMOKE_SAMPLES = 5
SMOKE_MAX_NEW_TOKENS = 32

if RUN_SMOKE:
    run_python(
        SCRIPT,
        "--tiny",
        "--samples",
        SMOKE_SAMPLES,
        "--max-new-tokens",
        SMOKE_MAX_NEW_TOKENS,
        "--device",
        "cuda",
    )
else:
    print("Smoke run skipped.")

## Real Qwen Comparison

After the smoke run succeeds, set `RUN_REAL = True` and run this cell. With 40 GB VRAM, the default 0.5B drafter plus 7B target in 4-bit should fit.

In [ ]:
RUN_REAL = False
REAL_SAMPLES = 50
REAL_MAX_NEW_TOKENS = 128

if RUN_REAL:
    run_python(
        SCRIPT,
        "--no-tiny",
        "--samples",
        REAL_SAMPLES,
        "--max-new-tokens",
        REAL_MAX_NEW_TOKENS,
        "--device",
        "cuda",
    )
else:
    print("Real run skipped. Set RUN_REAL = True when the smoke run is clean.")

## Regenerate Plots From Existing Results

Run this if the JSON files already exist and you only want to rebuild the CSV and PNG files.

In [ ]:
RUN_PLOT_ONLY = False

if RUN_PLOT_ONLY:
    run_python(SCRIPT, "--plot-only")
else:
    print("Plot-only run skipped.")

## View Merged Results

In [ ]:
csv_path = RESULT_DIR / "dynamic_k_comparison.csv"
if not csv_path.exists():
    print(f"No merged CSV yet: {csv_path}")
else:
    try:
        import pandas as pd

        display(pd.read_csv(csv_path))
    except ModuleNotFoundError:
        with csv_path.open("r", encoding="utf-8") as file:
            for row in csv.DictReader(file):
                print(row)

## View Plots

In [ ]:
plot_names = [
    "primary_metrics.png",
    "dynamic_k_summary.png",
    "dynamic_k_distribution.png",
    "controller_diagnostics.png",
]

for plot_name in plot_names:
    path = PLOTS_DIR / plot_name
    if path.exists():
        print(path)
        display(Image(filename=str(path)))
    else:
        print(f"Missing plot: {path}")